### Isolation Forest Experiment
This first experiment explores the performance of a simple isolation forest on the features used in RELAISS. We extract RELAISS features from codebase and explore how often the anomalies are in a low minority class. This indicates how the isolation forest is performing

Using 03_build_databank notebook to extract RELAISS features

**Goal:** Use Isolation Forest to detect anomalies in the dataset and interpret which observations are likely outliers.

**Isolation Forest:** It isolates points by randomly selecting features and split values; anomalies are easier to isolate (fewer split lines). 


In [1]:
import os
import pandas as pd
import numpy as np
from relaiss import constants
import relaiss as rl
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

In [2]:
csv_path = "/Users/jennakempster-taylor/re-laiss/reference_20k.csv" 

# Load
df = pd.read_csv(csv_path, low_memory=False)

# Ensure there is an object name column
if "objName" not in df.columns:
    # Some variants may include "NAME" or similar; map it if present
    for candidate in ["NAME", "name", "ObjName", "object_name"]:
        if candidate in df.columns:
            df = df.rename(columns={candidate: "objName"})
            break

if "objName" not in df.columns:
    raise ValueError("Could not find 'objName' in the CSV. ")

print("Shape:", df.shape)
print("Columns:", df.columns[:10])  # first 10 columns
print("First few objNames:", df["objName"].head())


Shape: (25515, 458)
Columns: Index(['t0', 'g_peak_mag', 'g_peak_time', 'g_rise_time', 'g_decline_time',
       'g_duration_above_half_flux', 'g_amplitude', 'g_skewness',
       'g_beyond_2sigma', 'r_peak_mag'],
      dtype='object')
First few objNames: 0    PSO J032.5595-01.8136
1    PSO J118.7364+35.9410
2    PSO J158.8836+37.6496
3    PSO J031.2821+11.2486
4    PSO J105.8641+32.0687
Name: objName, dtype: object


In [11]:
# Grab our RELAISS features
default_lc_features = constants.lc_features_const.copy()
default_host_features = constants.host_features_const.copy()

# quick look at what is included
print("Default LC features (sample):", default_lc_features[:10])
print("Default host features (sample):", default_host_features[:10])


Default LC features (sample): ['g_peak_time', 'r_peak_time', 'g_rise_time', 'g_decline_time', 'r_rise_time', 'r_decline_time', 'g_duration_above_half_flux', 'r_duration_above_half_flux', 'g_amplitude', 'r_amplitude']
Default host features (sample): ['gKronMagCorrected', 'gKronRad', 'gExtNSigma', 'rKronMagCorrected', 'rKronRad', 'rExtNSigma', 'iKronMagCorrected', 'iKronRad', 'iExtNSigma', 'zKronMagCorrected']


Double check the default RELAISS features are all in our csv file:

In [14]:
from pprint import pprint

USE_HOST = False  # use only light curves initially

def overlap(cols, frame):
    return [c for c in cols if (c in frame.columns) and (not c.endswith("_err"))]

lc_cols   = overlap(default_lc_features, df)
host_cols = overlap(default_host_features, df) if USE_HOST else []

feature_cols = lc_cols + host_cols

print(f"Found {len(lc_cols)} LC features in CSV.")
if USE_HOST:
    print(f"Found {len(host_cols)} host features in CSV.")
print(f"Total features used: {len(feature_cols)}")
pprint(feature_cols)


Found 25 LC features in CSV.
Total features used: 25
['g_peak_time',
 'r_peak_time',
 'g_rise_time',
 'g_decline_time',
 'r_rise_time',
 'r_decline_time',
 'g_duration_above_half_flux',
 'r_duration_above_half_flux',
 'g_amplitude',
 'r_amplitude',
 'g_skewness',
 'r_skewness',
 'g_beyond_2sigma',
 'r_beyond_2sigma',
 'mean_g-r',
 'g-r_at_g_peak',
 'mean_color_rate',
 'g_max_rolling_variance',
 'r_max_rolling_variance',
 'g_mean_rolling_variance',
 'r_mean_rolling_variance',
 'g_rise_local_curvature',
 'g_decline_local_curvature',
 'r_rise_local_curvature',
 'r_decline_local_curvature']


Start cleaning data: \
Might be missing or NaNs 

In [27]:
from sklearn.impute import SimpleImputer

X = df[feature_cols].copy()
X = X.replace([np.inf, -np.inf], np.nan)

# Quick look at missingness
missing_pct = X.isna().mean().sort_values(ascending=False)
print("Top 10 columns by missing rate:")
print(missing_pct.head(10))

#imp = SimpleImputer(strategy="median")
#X_imp = imp.fit_transform(X)

print("X shape:", X.shape, " -> after impute:", X_imp.shape)

Top 10 columns by missing rate:
g_rise_local_curvature        0.420419
r_rise_local_curvature        0.389183
r_decline_local_curvature     0.347443
g_decline_local_curvature     0.324005
g_rise_time                   0.264511
r_rise_time                   0.250872
r_decline_time                0.232491
g_decline_time                0.208818
r_duration_above_half_flux    0.152538
r_peak_time                   0.148893
dtype: float64
X shape: (25515, 25)  -> after impute: (25515, 25)


A lot of missing data here ~40%, so use KNN imputation. 

In [30]:
from sklearn.impute import KNNImputer

X = df[feature_cols].copy()
X = X.replace([np.inf, -np.inf], np.nan)

#  missing value is filled in using  mean of 5 nearest neighbors
knn_imp = KNNImputer(n_neighbors=5, weights="uniform")
X_imp = knn_imp.fit_transform(X)

print("X shape:", X.shape, " -> after KNN impute:", X_imp.shape)

X shape: (25515, 25)  -> after KNN impute: (25515, 25)


### Isolation Forest:

In [32]:
iso = IsolationForest(
    n_estimators=300,
    contamination="auto",   # use auto for first run
    random_state=42,
    n_jobs=-1
)
iso.fit(X_imp)

scores   = iso.decision_function(X_imp)  # higher = more normal
raw_pred = iso.predict(X_imp)            # -1 anomaly, 1 normal
anomaly  = (raw_pred == -1).astype(int)  # 1 = anomaly

rank = pd.Series(scores).rank(method="first", ascending=True).astype(int)  # 1 = most anomalous

print("Estimated anomaly rate:", anomaly.mean())

Estimated anomaly rate: 0.06274740348814423


In [33]:
out = pd.DataFrame({
    "ZTFID": df["ZTFID"].values,
    "r_duration_above_half_flux": df["r_duration_above_half_flux"].values,
    "iso_score": scores,
    "iso_anomaly": anomaly,
    "iso_rank": rank
    
})

# Optional context columns if present
context_cols = [c for c in ["t0","r_duration_above_half_flux", "g_peak_mag","r_peak_mag","mean_g-r","features_valid"] if c in df.columns]
preview = pd.concat([df[["ZTFID"] + context_cols].reset_index(drop=True),
                     out[["r_duration_above_half_flux","iso_score","iso_anomaly","iso_rank"]]], axis=1)

# Show 10 most anomalous
display(preview.sort_values("iso_rank").head(10))


,ZTFID,t0,r_duration_above_half_flux,g_peak_mag,r_peak_mag,mean_g-r,features_valid,r_duration_above_half_flux,iso_score,iso_anomaly,iso_rank
518,ZTF18ablqsov,58669.283854,1554.922824,16.045099,16.252300,-0.437946,False,1554.922824,-0.253933,1,1
185,ZTF18aalataf,58612.446134,1951.855185,16.848600,17.090679,-1.598087,False,1951.855185,-0.248612,1,2
544,ZTF18abnuekr,58644.460532,1953.669410,17.057899,17.020267,-0.460937,False,1953.669410,-0.247273,1,3
42,ZTF18aaapjfq,58360.514526,2429.651643,16.538094,17.262501,0.315912,False,2429.651643,-0.242008,1,4
529,ZTF18ablwvcy,58344.348553,2438.200417,17.004261,17.659700,0.103281,False,2438.200417,-0.235605,1,5
1068,ZTF18acxbexw,58557.360741,2199.962095,16.414274,16.488100,-0.120146,False,2199.962095,-0.233004,1,6
925,ZTF18acgvhxo,58435.521215,2333.845278,16.100832,15.568569,0.255854,False,2333.845278,-0.221767,1,7
225,ZTF18aanuzde,58252.491157,2533.963877,16.863199,17.648199,-0.439563,False,2533.963877,-0.219895,1,8
1448,ZTF19aawevyv,58634.426979,1763.092951,17.891172,17.878672,0.990386,False,1763.092951,-0.215577,1,9
63,ZTF18aabilqu,58360.513079,2208.998079,15.248654,14.825365,0.808560,True,2208.998079,-0.212265,1,10


In [34]:
# Extract name/id
row = df.loc[df["objName"] == "PSO J128.9117-04.0883", ["objName", "ZTFID"]]
print(row)
row = df.loc[df["objName"] == "PSO J283.6976+55.3976", ["objName", "ZTFID"]]
print(row)
row = df.loc[df["objName"] == "PSO J337.1748+43.2297", ["objName", "ZTFID"]]
print(row)
row = df.loc[df["objName"] == "PSO J115.6368+49.8097", ["objName", "ZTFID"]]
print(row)



                   objName         ZTFID
925  PSO J128.9117-04.0883  ZTF18acgvhxo
                   objName         ZTFID
225  PSO J283.6976+55.3976  ZTF18aanuzde
                    objName         ZTFID
1448  PSO J337.1748+43.2297  ZTF19aawevyv
                  objName         ZTFID
63  PSO J115.6368+49.8097  ZTF18aabilqu


Given that the top 10 'anomalies' are features_valid = false, then this is not a good sign. 
The features_valid flag is a RELAISS quality check: if it’s True, the features were reliably computed; if it’s False, some features were missing or unreliable.

In [36]:
# Split anomalies by features_valid

# Valid anomalies (features_valid == true)
valid_anomalies = (
    preview[preview["features_valid"] == True]
    .sort_values("iso_rank")
    .reset_index(drop=True)
)

# Unreliable anomalies 
invalid_anomalies = (
    preview[preview["features_valid"] == False]
    .sort_values("iso_rank")
    .reset_index(drop=True)
)

# Show top 10 from each
print("Top 10 valid anomalies:")
display(valid_anomalies.head(10))

print("Top 10 strange anomalies:")
display(invalid_anomalies.head(10))

# Save separately
valid_anomalies.to_csv("reference_20k_valid_anomalies.csv", index=False)
invalid_anomalies.to_csv("reference_20k_invalid_anomalies.csv", index=False)


Top 10 valid anomalies:


,ZTFID,t0,r_duration_above_half_flux,g_peak_mag,r_peak_mag,mean_g-r,features_valid,r_duration_above_half_flux,iso_score,iso_anomaly,iso_rank
0,ZTF18aabilqu,58360.513079,2208.998079,15.248654,14.825365,0.808560,True,2208.998079,-0.212265,1,10
1,ZTF18abjhbss,58322.424942,1913.823275,17.262716,16.456707,0.578618,True,1913.823275,-0.210216,1,12
2,ZTF18abtvajf,58368.229873,1561.892824,17.080242,16.883101,1.151374,True,1561.892824,-0.209298,1,14
3,ZTF18adisfiy,58252.413067,2062.203727,16.432600,16.021099,0.525252,True,2062.203727,-0.207879,1,16
4,ZTF19aadnhaw,58883.242176,1841.182951,15.795343,15.998410,-0.149093,True,1841.182951,-0.207361,1,17
5,ZTF18aaqzomm,58252.411609,2399.220775,17.181522,17.578285,-0.164648,True,2399.220775,-0.205367,1,18
6,ZTF18acvgzbp,58472.443588,1480.996991,14.932600,14.918000,0.058028,True,1480.996991,-0.203664,1,21
7,ZTF18abgoodz,58312.473831,2329.770058,16.842958,16.490053,0.367807,True,2329.770058,-0.200743,1,26
8,ZTF18abacxpl,58281.407951,1576.330938,17.313900,17.428900,-0.150926,True,1576.330938,-0.195991,1,30
9,ZTF20acxhear,58456.247141,2073.317847,17.401100,16.331100,1.483282,True,2073.317847,-0.195324,1,32


Top 10 strange anomalies:


,ZTFID,t0,r_duration_above_half_flux,g_peak_mag,r_peak_mag,mean_g-r,features_valid,r_duration_above_half_flux,iso_score,iso_anomaly,iso_rank
0,ZTF18ablqsov,58669.283854,1554.922824,16.045099,16.252300,-0.437946,False,1554.922824,-0.253933,1,1
1,ZTF18aalataf,58612.446134,1951.855185,16.848600,17.090679,-1.598087,False,1951.855185,-0.248612,1,2
2,ZTF18abnuekr,58644.460532,1953.669410,17.057899,17.020267,-0.460937,False,1953.669410,-0.247273,1,3
3,ZTF18aaapjfq,58360.514526,2429.651643,16.538094,17.262501,0.315912,False,2429.651643,-0.242008,1,4
4,ZTF18ablwvcy,58344.348553,2438.200417,17.004261,17.659700,0.103281,False,2438.200417,-0.235605,1,5
5,ZTF18acxbexw,58557.360741,2199.962095,16.414274,16.488100,-0.120146,False,2199.962095,-0.233004,1,6
6,ZTF18acgvhxo,58435.521215,2333.845278,16.100832,15.568569,0.255854,False,2333.845278,-0.221767,1,7
7,ZTF18aanuzde,58252.491157,2533.963877,16.863199,17.648199,-0.439563,False,2533.963877,-0.219895,1,8
8,ZTF19aawevyv,58634.426979,1763.092951,17.891172,17.878672,0.990386,False,1763.092951,-0.215577,1,9
9,ZTF18aagpxsx,58471.276400,2215.982072,16.901436,17.032700,-0.094638,False,2215.982072,-0.210590,1,11


The following blocks are looking at number of anomalies found to see what is going on with features_valid

In [38]:
print(out["iso_anomaly"].value_counts())
print(out["iso_anomaly"].value_counts(normalize=True))  # fraction


iso_anomaly
0    23914
1     1601
Name: count, dtype: int64
iso_anomaly
0    0.937253
1    0.062747
Name: proportion, dtype: float64


In [39]:
print(pd.crosstab(preview["features_valid"], preview["iso_anomaly"]))


iso_anomaly         0     1
features_valid             
False           17715  1105
True             6199   496


In [40]:
if "best_cat" in df.columns:
    print(pd.crosstab(df["best_cat"], out["iso_anomaly"]))


iso_anomaly      0     1
best_cat                
panstarrs    20253  1554


In [41]:
print([c for c in df.columns if "tns" in c.lower() or "cat" in c.lower()])


['best_cat', 'best_cat_release', 'missedcat_posterior', 'extra_cat_cols', 'iExtNSigma', 'gExtNSigma', 'rExtNSigma', 'yExtNSigma', 'zExtNSigma', 'tns_redshift']


In [42]:
if "best_cat" in df.columns:
    print(pd.crosstab(df["best_cat"], out["iso_anomaly"]))


iso_anomaly      0     1
best_cat                
panstarrs    20253  1554


### Isolation Forest with different contamination:

In [53]:
iso = IsolationForest(
    n_estimators=300,
    contamination=0.1,   # use auto for first run
    random_state=42,
    n_jobs=-1
)
iso.fit(X_imp)

scores   = iso.decision_function(X_imp)  # higher = more normal
raw_pred = iso.predict(X_imp)            # -1 anomaly, 1 normal
anomaly  = (raw_pred == -1).astype(int)  # 1 = anomaly

rank = pd.Series(scores).rank(method="first", ascending=True).astype(int)  # 1 = most anomalous

print("Estimated anomaly rate:", anomaly.mean())

Estimated anomaly rate: 0.1000195963158926


In [54]:
out = pd.DataFrame({
    "objName": df["objName"].values,
    "iso_score": scores,
    "iso_anomaly": anomaly,
    "iso_rank": rank
})

# Optional context columns if present
context_cols = [c for c in ["t0","g_peak_mag","r_peak_mag","mean_g-r","features_valid"] if c in df.columns]
preview = pd.concat([df[["objName"] + context_cols].reset_index(drop=True),
                     out[["iso_score","iso_anomaly","iso_rank"]]], axis=1)

# Show 10 most anomalous
display(preview.sort_values("iso_rank").head(10))


,objName,t0,g_peak_mag,r_peak_mag,mean_g-r,features_valid,iso_score,iso_anomaly,iso_rank
518,PSO J275.9690+23.5268,58669.283854,16.045099,16.252300,-0.437946,False,-0.312751,1,1
185,PSO J275.6485+41.8038,58612.446134,16.848600,17.090679,-1.598087,False,-0.307429,1,2
544,PSO J303.4828-05.4350,58644.460532,17.057899,17.020267,-0.460937,False,-0.306090,1,3
42,PSO J109.8504+29.3950,58360.514526,16.538094,17.262501,0.315912,False,-0.300825,1,4
529,PSO J289.7529+36.0261,58344.348553,17.004261,17.659700,0.103281,False,-0.294422,1,5
1068,PSO J213.7545-09.2659,58557.360741,16.414274,16.488100,-0.120146,False,-0.291821,1,6
925,PSO J128.9117-04.0883,58435.521215,16.100832,15.568569,0.255854,False,-0.280584,1,7
225,PSO J283.6976+55.3976,58252.491157,16.863199,17.648199,-0.439563,False,-0.278712,1,8
1448,PSO J337.1748+43.2297,58634.426979,17.891172,17.878672,0.990386,False,-0.274395,1,9
63,PSO J115.6368+49.8097,58360.513079,15.248654,14.825365,0.808560,True,-0.271082,1,10


In [57]:
# Split anomalies by features_valid

# Valid anomalies (features_valid == true)
valid_anomalies = (
    preview[preview["features_valid"] == True]
    .sort_values("iso_rank")
    .reset_index(drop=True)
)

# Unreliable anomalies 
invalid_anomalies = (
    preview[preview["features_valid"] == False]
    .sort_values("iso_rank")
    .reset_index(drop=True)
)

# Show top 10 from each
print("Top 10 valid anomalies:")
display(valid_anomalies.head(10))

print("Top 10 strange anomalies:")
display(invalid_anomalies.head(10))

# Save separately
valid_anomalies.to_csv("reference_20k_valid_anomalies.csv", index=False)
invalid_anomalies.to_csv("reference_20k_invalid_anomalies.csv", index=False)


Top 10 valid anomalies:


,objName,t0,g_peak_mag,r_peak_mag,mean_g-r,features_valid,iso_score,iso_anomaly,iso_rank
0,PSO J115.6368+49.8097,58360.513079,15.248654,14.825365,0.808560,True,-0.271082,1,10
1,PSO J307.0327+22.3902,58322.424942,17.262716,16.456707,0.578618,True,-0.269033,1,12
2,PSO J292.0967-13.3364,58368.229873,17.080242,16.883101,1.151374,True,-0.268115,1,14
3,PSO J278.5756+31.6068,58252.413067,16.432600,16.021099,0.525252,True,-0.266696,1,16
4,PSO J152.9325+57.3038,58883.242176,15.795343,15.998410,-0.149093,True,-0.266178,1,17
5,PSO J284.6753+44.0967,58252.411609,17.181522,17.578285,-0.164648,True,-0.264184,1,18
6,PSO J155.4177-03.4538,58472.443588,14.932600,14.918000,0.058028,True,-0.262481,1,21
7,PSO J010.9273+37.4222,58312.473831,16.842958,16.490053,0.367807,True,-0.259561,1,26
8,PSO J293.8673+53.7628,58281.407951,17.313900,17.428900,-0.150926,True,-0.254808,1,30
9,PSO J092.9730-06.1507,58456.247141,17.401100,16.331100,1.483282,True,-0.254141,1,32


Top 10 strange anomalies:


,objName,t0,g_peak_mag,r_peak_mag,mean_g-r,features_valid,iso_score,iso_anomaly,iso_rank
0,PSO J275.9690+23.5268,58669.283854,16.045099,16.252300,-0.437946,False,-0.312751,1,1
1,PSO J275.6485+41.8038,58612.446134,16.848600,17.090679,-1.598087,False,-0.307429,1,2
2,PSO J303.4828-05.4350,58644.460532,17.057899,17.020267,-0.460937,False,-0.306090,1,3
3,PSO J109.8504+29.3950,58360.514526,16.538094,17.262501,0.315912,False,-0.300825,1,4
4,PSO J289.7529+36.0261,58344.348553,17.004261,17.659700,0.103281,False,-0.294422,1,5
5,PSO J213.7545-09.2659,58557.360741,16.414274,16.488100,-0.120146,False,-0.291821,1,6
6,PSO J128.9117-04.0883,58435.521215,16.100832,15.568569,0.255854,False,-0.280584,1,7
7,PSO J283.6976+55.3976,58252.491157,16.863199,17.648199,-0.439563,False,-0.278712,1,8
8,PSO J337.1748+43.2297,58634.426979,17.891172,17.878672,0.990386,False,-0.274395,1,9
9,PSO J112.5001+11.0348,58471.276400,16.901436,17.032700,-0.094638,False,-0.269408,1,11


In [59]:
print(out["iso_anomaly"].value_counts())
print(out["iso_anomaly"].value_counts(normalize=True))  # fraction


iso_anomaly
0    22963
1     2552
Name: count, dtype: int64
iso_anomaly
0    0.89998
1    0.10002
Name: proportion, dtype: float64


In [61]:
df["best_cat_release"].value_counts().head(20)


best_cat_release
dr2    21807
Name: count, dtype: int64

In [63]:
[c for c in df.columns if "tns" in c.lower() or "class" in c.lower()]


['iExtNSigma',
 'gExtNSigma',
 'rExtNSigma',
 'yExtNSigma',
 'zExtNSigma',
 'tns_redshift']

As we are getting quasars and AGNs, we need to refine features to target anomalous supernovae. Here we will inspect features used and see what we can remove by hand. 

In [66]:
# List column names
headers = df.columns.tolist()
print(headers)

# Just see the first 20 headers
print(df.columns[:20])

# columns total
print("Number of columns:", len(df.columns))


['t0', 'g_peak_mag', 'g_peak_time', 'g_rise_time', 'g_decline_time', 'g_duration_above_half_flux', 'g_amplitude', 'g_skewness', 'g_beyond_2sigma', 'r_peak_mag', 'r_peak_time', 'r_rise_time', 'r_decline_time', 'r_duration_above_half_flux', 'r_amplitude', 'r_skewness', 'r_beyond_2sigma', 'mean_g-r', 'g-r_at_g_peak', 'mean_color_rate', 'g_n_peaks', 'g_dt_main_to_secondary_peak', 'g_dmag_secondary_peak', 'g_secondary_peak_prominence', 'g_secondary_peak_width', 'r_n_peaks', 'r_dt_main_to_secondary_peak', 'r_dmag_secondary_peak', 'r_secondary_peak_prominence', 'r_secondary_peak_width', 'g_max_rolling_variance', 'g_mean_rolling_variance', 'r_max_rolling_variance', 'r_mean_rolling_variance', 'g_rise_local_curvature', 'g_decline_local_curvature', 'r_rise_local_curvature', 'r_decline_local_curvature', 'features_valid', 't0_err', 'g_peak_mag_err', 'g_peak_time_err', 'g_rise_time_err', 'g_decline_time_err', 'g_duration_above_half_flux_err', 'g_amplitude_err', 'g_skewness_err', 'g_beyond_2sigma_err

Now, get rid of lc with r_duration_above_half_flux longer than 2 years. r_duration_above_half_flux > 781

In [69]:
csv_path = "/Users/jennakempster-taylor/re-laiss/reference_20k.csv" 

# Load
df = pd.read_csv(csv_path, low_memory=False)

# Ensure there is an object name column
if "objName" not in df.columns:
    # Some variants may include "NAME" or similar,, map it if present
    for candidate in ["NAME", "name", "ObjName", "object_name"]:
        if candidate in df.columns:
            df = df.rename(columns={candidate: "objName"})
            break

if "objName" not in df.columns:
    raise ValueError("Could not find 'objName' in the CSV. ")

print("Shape:", df.shape)
print("Columns:", df.columns[:10])  # first 10 columns
print("First few objNames:", df["objName"].head())

default_lc_features = constants.lc_features_const.copy()
default_host_features = constants.host_features_const.copy()

from pprint import pprint

USE_HOST = False  # use only light curves initially

def overlap(cols, frame):
    return [c for c in cols if (c in frame.columns) and (not c.endswith("_err"))]

lc_cols   = overlap(default_lc_features, df)
host_cols = overlap(default_host_features, df) if USE_HOST else []

feature_cols = lc_cols + host_cols

print(f"Found {len(lc_cols)} LC features in CSV.")
if USE_HOST:
    print(f"Found {len(host_cols)} host features in CSV.")
print(f"Total features used: {len(feature_cols)}")
pprint(feature_cols[:15])

from sklearn.impute import SimpleImputer

X = df[feature_cols].copy()
X = X.replace([np.inf, -np.inf], np.nan)

# Quick look at missingness
missing_pct = X.isna().mean().sort_values(ascending=False)
print("Top 10 columns by missing rate:")
print(missing_pct.head(10))

#imp = SimpleImputer(strategy="median")
#X_imp = imp.fit_transform(X)

print("X shape:", X.shape, " -> after impute:", X_imp.shape)

from sklearn.impute import KNNImputer

X = df[feature_cols].copy()
X = X.replace([np.inf, -np.inf], np.nan)

#  missing value is filled in using  mean of 5 nearest neighbors
knn_imp = KNNImputer(n_neighbors=5, weights="uniform")
X_imp = knn_imp.fit_transform(X)

print("X shape:", X.shape, " -> after KNN impute:", X_imp.shape)




Shape: (25515, 458)
Columns: Index(['t0', 'g_peak_mag', 'g_peak_time', 'g_rise_time', 'g_decline_time',
       'g_duration_above_half_flux', 'g_amplitude', 'g_skewness',
       'g_beyond_2sigma', 'r_peak_mag'],
      dtype='object')
First few objNames: 0    PSO J032.5595-01.8136
1    PSO J118.7364+35.9410
2    PSO J158.8836+37.6496
3    PSO J031.2821+11.2486
4    PSO J105.8641+32.0687
Name: objName, dtype: object
Found 25 LC features in CSV.
Total features used: 25
['g_peak_time',
 'r_peak_time',
 'g_rise_time',
 'g_decline_time',
 'r_rise_time',
 'r_decline_time',
 'g_duration_above_half_flux',
 'r_duration_above_half_flux',
 'g_amplitude',
 'r_amplitude',
 'g_skewness',
 'r_skewness',
 'g_beyond_2sigma',
 'r_beyond_2sigma',
 'mean_g-r']
Top 10 columns by missing rate:
g_rise_local_curvature        0.420419
r_rise_local_curvature        0.389183
r_decline_local_curvature     0.347443
g_decline_local_curvature     0.324005
g_rise_time                   0.264511
r_rise_time             

In [70]:
# Filter out non-SN lightcurves

DUR_CUTOFF = 731  # days

if "r_duration_above_half_flux" not in df.columns:
    raise KeyError("r_duration_above_half_flux not found in df.columns")

n_before = len(df)
# mask_keep = (df["r_duration_above_half_flux"].notna()) & (df["r_duration_above_half_flux"] <= DUR_CUTOFF)
# If you also want to keep rows with NaN durations, change to:
mask_keep = (df["r_duration_above_half_flux"].isna()) | (df["r_duration_above_half_flux"] <= DUR_CUTOFF)

df_filt = df[mask_keep].copy()
n_after = len(df_filt)
print(f"Filtered by r_duration_above_half_flux <= {DUR_CUTOFF} days: kept {n_after}/{n_before} "
      f"({n_after/n_before:.1%}); removed {n_before-n_after}")

# save the removed objects for audit
removed = df[~mask_keep][["objName","ZTFID","r_duration_above_half_flux"]].copy()
removed.to_csv("removed_long_duration_objects.csv", index=False)
print("Saved removed_long_duration_objects.csv")


Filtered by r_duration_above_half_flux <= 731 days: kept 24277/25515 (95.1%); removed 1238
Saved removed_long_duration_objects.csv


In [ ]:
# Use df_filt from here on 
# 1 Features on the filtered frame
lc_cols   = [c for c in default_lc_features if (c in df_filt.columns) and (not c.endswith("_err"))]
host_cols = []  # add host features later if you like
feature_cols = lc_cols + host_cols

X = df_filt[feature_cols].replace([np.inf, -np.inf], np.nan)

# 2 Impute (KNN) on the FILTERED X
from sklearn.impute import KNNImputer
knn_imp = KNNImputer(n_neighbors=5, weights="uniform")
X_imp = knn_imp.fit_transform(X)

print("Shapes -> df_filt:", df_filt.shape, " X:", X.shape, " X_imp:", X_imp.shape)

# 3 Isolation Forest on the FILTERED, IMPUTED matrix
from sklearn.ensemble import IsolationForest
iso = IsolationForest(n_estimators=300, contamination="auto", random_state=42, n_jobs=-1)
iso.fit(X_imp)

import pandas as pd
scores   = iso.decision_function(X_imp)   # length == len(df_filt)
raw_pred = iso.predict(X_imp)
anomaly  = (raw_pred == -1).astype(int)
rank     = pd.Series(scores, index=df_filt.index).rank(method="first", ascending=True).astype(int)

# 4 Pack results (ensure lengths match)
assert len(scores) == len(df_filt), "Length mismatch: scores vs df_filt"

out = pd.DataFrame({
    "ZTFID":      df_filt["ZTFID"].values,
    "iso_score":  scores,
    "iso_anomaly": anomaly,
    "iso_rank":   rank.values
}, index=df_filt.index)

# 5 Build preview from the FILTERED df only
context_cols = [c for c in ["t0","r_duration_above_half_flux","g_peak_mag","r_peak_mag","mean_g-r","features_valid","ZTFID"] 
                if c in df_filt.columns]

preview = pd.concat(
    [df_filt[["ZTFID"] + context_cols], out[["iso_score","iso_anomaly","iso_rank"]]],
    axis=1
)

display(preview.sort_values("iso_rank").head(10))

# sanity checks
print("\nAnomaly rate:", out["iso_anomaly"].mean())
print("Value counts:\n", out["iso_anomaly"].value_counts())



In [ ]:
# All column headers
all_cols = df.columns.tolist()
print("Number of columns:", len(all_cols))
print("First 20 columns:", all_cols[:20])


In [ ]:
pd.Series(df.columns).to_csv("reference_20k_headers.csv", index=False)
print("Saved headers to reference_20k_headers.csv")


In [ ]:

from scipy.stats import spearmanr

feat_df = df_filt[feature_cols].copy()
feat_df = feat_df.replace([np.inf,-np.inf], np.nan)
feat_df = pd.DataFrame(X_imp, columns=feature_cols, index=df_filt.index)  # imputed version

def spearman_to_score(col):
    v = feat_df[col].values
    m = np.isfinite(v) & np.isfinite(scores)
    r, p = spearmanr(v[m], scores[m])
    return r, p

corrs = pd.DataFrame({f: spearman_to_score(f) for f in feature_cols}, index=["rho","p"]).T
corrs_sorted = corrs.sort_values("rho")  # most negative -> more anomalous when feature is large
corrs_sorted.head(15)


In [ ]:
transient_features = [
    "g_peak_mag","r_peak_mag",
    "g_amplitude","r_amplitude",
    "g_rise_time","r_rise_time",
    "g_decline_time","r_decline_time",
    "g_skewness","r_skewness",
    "g_n_peaks","r_n_peaks",
    "mean_g-r","g-r_at_g_peak","mean_color_rate",
    "g_beyond_2sigma","r_beyond_2sigma"
]

# Keep only features that actually exist in your df
transient_cols = [c for c in transient_features if c in df_filt.columns]

print("Transient-focused features:", len(transient_cols))
print(transient_cols)

In [ ]:


X_trans = df_filt[transient_cols].replace([np.inf, -np.inf], np.nan)

from sklearn.impute import KNNImputer
knn_imp = KNNImputer(n_neighbors=5)
X_trans_imp = knn_imp.fit_transform(X_trans)

# Train IF 
from sklearn.ensemble import IsolationForest
iso_trans = IsolationForest(n_estimators=300, contamination="auto", random_state=42, n_jobs=-1)
iso_trans.fit(X_trans_imp)

scores_trans   = iso_trans.decision_function(X_trans_imp)
raw_pred_trans = iso_trans.predict(X_trans_imp)
anomaly_trans  = (raw_pred_trans == -1).astype(int)

import pandas as pd
rank_trans = pd.Series(scores_trans, index=df_filt.index).rank(method="first", ascending=True).astype(int)

out_trans = pd.DataFrame({
    "ZTFID": df_filt["ZTFID"].values,
    "iso_score": scores_trans,
    "iso_anomaly": anomaly_trans,
    "iso_rank": rank_trans.values
}, index=df_filt.index)

preview_trans = pd.concat([
    df_filt[["objName","ZTFID","features_valid"]],
    out_trans[["iso_score","iso_anomaly","iso_rank"]]
], axis=1)

display(preview_trans.sort_values("iso_rank").head(10))

We will continue to explore what features are needed after looking at the results of this run. It is likely that more refinement is required. 